# FINAL ONE-SHOT OOD BENCHMARK

**This notebook spends something that cannot be refilled.**

It evaluates the already-selected deployment candidate - the INT8 ONNX graph
accepted by [`model_v2_deberta_v3_xsmall.ipynb`](model_v2_deberta_v3_xsmall.ipynb) -
on the frozen OOD final benchmark, exactly once.

**Do not run this notebook:**

* during training;
* during model selection;
* to tune thresholds;
* to choose augmentation;
* to compare unfinished checkpoints;
* repeatedly while developing.

Running this benchmark exposes the final holdout. Once its results influence any
future model change, this benchmark is no longer pristine for that next
model-generation cycle.

Build and accept a candidate in the main notebook first. That notebook is safe to
**Run All**; this one is not.


## What this notebook does

```
verify accepted candidate -> verify benchmark identity
  -> load the accepted INT8 model, tokenizer and label map
  -> load the frozen OOD final fixture -> evaluate once -> save the report
```

It does not train, build datasets, augment, select checkpoints, quantize, tune
thresholds, run OOD dev, or modify the candidate in any way.

It scores the accepted **INT8 ONNX candidate and nothing else** - no PyTorch, no
FP32 ONNX, no TinyBERT, no alternate checkpoint. The point is to measure the one
artifact that passed release selection. A later TinyBERT comparison must use the
existing historical results, not a second pass over this holdout.


## 1. Environment, paths, and the rerun override


In [ ]:
import json, os, sys, time
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "FullPipeline_V3").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "FullPipeline_V3").is_dir():
    raise RuntimeError("Run this notebook from inside the OnKith repository.")
for extra in (REPO_ROOT / "model" / "privacy_data_v2", REPO_ROOT / "model" / "privacy_evaluation"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from privacy_eval import accepted_candidate

# --- the one-shot override --------------------------------------------------
# Leave this False. Setting it True re-spends a benchmark that has already been
# spent, and is only defensible when recovering a report lost after a run that
# genuinely happened. It is not a testing switch.
ALLOW_FROZEN_RERUN = False

MODEL_DIR = REPO_ROOT / "model" / "DeBERTa-v3-xsmall"
METRICS_DIR = MODEL_DIR / "artifacts" / "model_v2" / "metrics"
CANDIDATE_MANIFEST_PATH = METRICS_DIR / accepted_candidate.CANDIDATE_MANIFEST_NAME
REPORT_PATH = METRICS_DIR / accepted_candidate.FROZEN_REPORT_NAME

FIXTURES = REPO_ROOT / "model" / "privacy_evaluation" / "fixtures"
OOD_FINAL_PATH = FIXTURES / "ood_final.jsonl"
OOD_FINAL_MANIFEST_PATH = FIXTURES / "ood_final.manifest.json"

# Smoke mode is a development mode. It redirects artifacts, bounds evaluation and
# produces nothing releasable, so it must never reach this notebook.
if os.environ.get("MODEL_V2_SMOKE_TEST", "0") == "1":
    raise RuntimeError(
        "MODEL_V2_SMOKE_TEST=1: smoke runs never spend the frozen OOD final benchmark. "
        "Unset the variable and use a real accepted candidate."
    )

print("repo root        :", REPO_ROOT)
print("candidate manifest:", CANDIDATE_MANIFEST_PATH.relative_to(REPO_ROOT))
print("report           :", REPORT_PATH.relative_to(REPO_ROOT))
print("frozen fixture   :", OOD_FINAL_PATH.relative_to(REPO_ROOT))
print("ALLOW_FROZEN_RERUN:", ALLOW_FROZEN_RERUN)


## 2. Verify the accepted candidate

Everything downstream trusts `accepted_int8_candidate.json` and nothing else, so
it is checked hard before the fixture is opened. `load_accepted_candidate` raises
on the first inconsistency: a missing manifest, `accepted != true`, a smoke
origin, a missing INT8 file, an INT8 SHA-256 that no longer matches, a label
count that is not 63, a changed ontology or label-map hash, a changed
tokenizer/config identity, a run signature that disagrees with the training
config, missing normal-evaluation evidence, or an INT8 gate that did not pass.

The SHA-256 re-check is what stops a hand-copied ONNX graph from quietly taking
the accepted candidate's place.


In [ ]:
candidate = accepted_candidate.load_accepted_candidate(
    CANDIDATE_MANIFEST_PATH, repo_root=REPO_ROOT
)
manifest = candidate["manifest"]

print("accepted candidate verified")
print("  int8            :", manifest["model"]["int8_onnx"])
print("  int8 sha256     :", candidate["int8_sha256"])
print("  int8 bytes      :", f"{manifest['model']['int8_bytes']:,}")
print("  ontology        :", manifest["ontology"]["version"],
      "|", manifest["ontology"]["num_entities"], "entities /",
      manifest["ontology"]["num_labels"], "labels")
print("  label map sha256:", manifest["ontology"]["label_map_sha256"][:16], "...")
print("  tokenizer       :", manifest["tokenizer"]["path"],
      "|", manifest["tokenizer"]["base_model_revision"][:12], "...")
print("  run signature   :", manifest["training"]["run_signature"][:16], "...")
print("  best epoch      :", manifest["training"]["best_epoch"],
      "| combined typed F1", f"{manifest['training']['best_combined_typed_f1']:.4f}")
print("  built           :", manifest["generated_utc"])

print("\nINT8 quality gate as recorded at acceptance:")
for view, entry in manifest["int8_gate"]["results"].items():
    print(f"  {view:20} dF1 {entry['typed_f1_drop']:>+8.4f} | "
          f"d leak {entry['leakage_rise']:>+8.4f} | "
          f"{'PASS' if entry['passes'] else 'FAIL'}")

print("\nINT8 evaluation evidence carried into this benchmark:")
for view, entry in manifest["evaluation"]["int8_onnx"].items():
    print(f"  {view:20} typed F1 {entry['typed_entity_f1']:.4f} | "
          f"leakage {entry['private_character_leakage']:.4f}")


## 3. Verify the frozen benchmark identity

The fixture is checked against its committed manifest **and** against the
fingerprint the main notebook recorded when this candidate was built, so a
benchmark that changed since then cannot be scored as if it had not.

Only hashes, counts and key names are inspected. No case text and no annotation
is read here.


In [ ]:
recorded_fixture = manifest["frozen_benchmark"]["fixture"]
fixture = accepted_candidate.verify_frozen_fixture(
    OOD_FINAL_PATH, OOD_FINAL_MANIFEST_PATH,
    expected_sha256=recorded_fixture.get("sha256"),
    expected_case_count=recorded_fixture.get("case_count"),
)

print("frozen fixture verified")
print("  path            :", OOD_FINAL_PATH.relative_to(REPO_ROOT))
print("  sha256          :", fixture["sha256"])
print("  cases           :", fixture["case_count"])
print("  hard negatives  :", fixture["hard_negative_count"])
print("  generator       :", fixture["generator_version"],
      "| schema", fixture["schema_version"])
print("  categories      :", ", ".join(
    f"{name} {count}" for name, count in sorted(fixture["category_allocation"].items())))
print("\npolicy:", fixture["policy"])


## 4. One-shot guard

If a report already exists for this fixture the notebook stops. Re-running on the
same candidate produces no new information and normalises spending the holdout;
running a *second* candidate is worse, because it turns the final set into a
selection metric.

`ALLOW_FROZEN_RERUN` exists only to recover a lost report for a run that already
happened. Leave it `False`.


In [ ]:
accepted_candidate.guard_one_shot(
    REPORT_PATH,
    candidate_sha256=candidate["int8_sha256"],
    fixture_sha256=fixture["sha256"],
    allow_rerun=ALLOW_FROZEN_RERUN,
)
if ALLOW_FROZEN_RERUN:
    print("WARNING: ALLOW_FROZEN_RERUN is True. The one-shot guard is disabled and an")
    print("existing report may be overwritten. This is only for archival recovery.")
elif REPORT_PATH.exists():
    print("A report exists for a different fixture; this fixture has not been spent.")
else:
    print("No existing report for this fixture. The benchmark has not been spent yet.")
print("\nProceeding to score the frozen set once.")


## 5. Score the accepted INT8 candidate, once

One runtime, one pass. The candidate is loaded through the same
`ModelV2OnnxMasker` decoding path the deployed pipeline uses, so this measures
deployed behaviour rather than a checkpoint nobody releases.


In [ ]:
import onnxruntime as ort
from records import from_fixture_case
from privacy_eval.dataset_metrics import evaluate_spans, password_analysis, subset_by
from privacy_eval.model_v2_adapter import ModelV2OnnxMasker
from privacy_eval.schema import load_cases

int8_masker = ModelV2OnnxMasker(
    candidate["int8_path"], candidate["tokenizer_dir"], candidate["label_map_path"],
    context_limit=manifest["training"]["context_limit"],
    overflow_stride=manifest["training"]["overflow_stride"],
)
assert int8_masker.num_labels == manifest["ontology"]["num_labels"], (
    int8_masker.num_labels, manifest["ontology"]["num_labels"]
)

ood_final = [from_fixture_case(case.to_dict(), partition="ood_final")
             for case in load_cases(OOD_FINAL_PATH)]
assert len(ood_final) == fixture["case_count"], (len(ood_final), fixture["case_count"])

started = time.perf_counter()
predictions = []
for record in ood_final:
    result = int8_masker.mask_text(record["source_text"])
    predictions.append([
        {"label": entity["entity_type"], "start": entity["start"], "end": entity["end"]}
        for entity in result.entities
    ])
elapsed = time.perf_counter() - started

ood_final_metrics = evaluate_spans(ood_final, predictions)
ood_final_by_category = subset_by(ood_final, predictions, "category")
ood_final_password = password_analysis(ood_final, predictions)

print(f"scored {len(ood_final)} frozen cases in {elapsed:.1f}s "
      f"({len(ood_final) / elapsed:.1f} cases/s)\n")
print(f"  typed entity F1                   {ood_final_metrics['typed_entity']['f1']:.4f}")
print(f"  binary entity F1                  {ood_final_metrics['binary_entity']['f1']:.4f}")
print(f"  private character recall          {ood_final_metrics['private_character_recall']:.4f}")
print(f"  private character leakage         {ood_final_metrics['private_character_leakage']:.4f}")
print(f"  non-private overmasking rate      {ood_final_metrics['non_private_overmasking_rate']:.4f}")
print(f"  hard-negative false positive rate {ood_final_metrics['hard_negative_false_positive_rate']:.4f}")

print("\nby category:")
for category, metrics in sorted(ood_final_by_category.items()):
    print(f"  {category:26} typed F1 {metrics['typed_entity']['f1']:.4f} | "
          f"char recall {metrics['private_character_recall']:.4f} | "
          f"hard-neg FP {metrics['hard_negative_false_positive_rate']:.4f}")

print("\nTinyBERT production reference, final partition (historical, not re-run here):")
print("  typed F1 0.5665 | char recall 0.6819 | leakage 0.3181 | hard-neg FP 0.6528")
print("Model V2 numbers are model-only, so the fairer TinyBERT comparison is its")
print("model-only INT8 typed F1 of 0.4649.")


## 6. Benchmark report

`evaluation_ood_final_int8.json` records the numbers together with every identity
that produced them: the candidate INT8 hash, the accepted-candidate manifest
hash, ontology and tokenizer identity, the frozen fixture hash and case count,
the runtime, and a UTC timestamp.

This is a measurement, not an input. Do not tune anything against it.


In [ ]:
report = accepted_candidate.build_frozen_report(
    repo_root=REPO_ROOT,
    candidate=candidate,
    fixture=fixture,
    metrics=ood_final_metrics,
    by_category=ood_final_by_category,
    password=ood_final_password,
    runtime={
        "runtime": "onnx_int8",
        "execution_provider": "CPUExecutionProvider",
        "onnxruntime_version": ort.__version__,
        "available_providers": ort.get_available_providers(),
        "context_limit": manifest["training"]["context_limit"],
        "overflow_stride": manifest["training"]["overflow_stride"],
        "evaluation_seconds": elapsed,
        "cases_per_second": len(ood_final) / elapsed,
    },
    timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
)
REPORT_PATH.write_text(json.dumps(report, indent=2, default=float) + "\n", encoding="utf-8")

print("wrote", REPORT_PATH.relative_to(REPO_ROOT))
print("  candidate int8 sha256:", report["candidate"]["int8_sha256"])
print("  candidate manifest sha256:", report["candidate"]["accepted_candidate_manifest_sha256"])
print("  fixture sha256       :", report["fixture"]["sha256"],
      f"({report['fixture']['case_count']} cases)")
print("  generated            :", report["generated_utc"])
print("\nThe frozen OOD final benchmark is now spent on this candidate.")
print("Re-running this notebook will refuse. Do not score a tuned variant.")


---

## Done

`artifacts/model_v2/metrics/evaluation_ood_final_int8.json` is the release
evidence for this model generation. It is a measurement of the accepted INT8
candidate, taken once.

From here:

* quote these numbers in the release record;
* do **not** feed them back into data, augmentation, thresholds or checkpoint
  selection for this generation;
* if a next model generation needs a pristine final benchmark, build a new frozen
  fixture rather than re-using this one.

Raspberry Pi latency, RAM and thermal behaviour remain separate deployment gates.
Nothing measured here is a Pi result.
